<a href="https://colab.research.google.com/github/TRPG-masterbot/data/blob/main/Jang/Input_Code_0.1_ver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
## RAG 불러오기
import requests
from pathlib import Path

import ast

# 저장소 경로
BASE_URL = "https://raw.githubusercontent.com/TRPG-masterbot/data/main"

# 모듈
MODULES = {
    "rule_rag.py": f"{BASE_URL}/Min/rule_rag.py",
    "sce_rag.py": f"{BASE_URL}/Kim/sce_rag.py"
}

SAVE_DIR = Path("rag")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

for filename, url in MODULES.items():

    response = requests.get(url, timeout=60)
    response.raise_for_status()

    save_path = SAVE_DIR / filename
    save_path.write_bytes(response.content)

    print("다운로드 완료:", filename)

다운로드 완료: rule_rag.py
다운로드 완료: sce_rag.py


In [2]:
## Chromadb 설치
%pip install -q chromadb

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
bigframes 2.51.0 r

In [3]:
import sys
from pathlib import Path

RAG_DIR = Path("rag").resolve()

if str(RAG_DIR) not in sys.path:
    sys.path.insert(0, str(RAG_DIR))

import rule_rag
import sce_rag

print("RULE RAG import 성공")
print("SCENARIO RAG import 성공")

RULE RAG import 성공
SCENARIO RAG import 성공


In [4]:
import inspect

print("===== RULE RAG 초기화 =====")
print(inspect.signature(rule_rag.load))
print(inspect.getsource(rule_rag.load))

print("\n===== SCENARIO RAG 초기화 =====")
print(inspect.signature(sce_rag.load))
print(inspect.getsource(sce_rag.load))

===== RULE RAG 초기화 =====
()
def load():                         # 검색 준비 (한 번만 부르면 돼요): 받기 → 청크 → 임베딩 → Chroma → BM25 → 조각 모음
    global chunks, collection, bm25, parts

    # 1. 깃허브에서 받기 (매번 최신으로)
    download("rules_new.jsonl")

    # 2. 청크 읽기 (규칙 청크만)
    chunks = []
    for line in open(OUT_PATH, encoding="utf-8"):
        c = json.loads(line)
        if c["meta"]["type"] == "rule":
            chunks.append(c)
    print("검색 대상 청크:", len(chunks))

    # 3. 임베딩: 저장된 게 있고 청크 수도 같으면 불러오기
    #    (청크 내용이 바뀌었는데 수가 같으면 옛 임베딩을 씀 → 그럴 땐 EMB_PATH 파일을 지우고 다시 load)
    saved = np.load(EMB_PATH) if os.path.exists(EMB_PATH) else None
    if saved is not None and saved.shape[0] == len(chunks):
        emb = saved
        print("저장된 임베딩 사용:", emb.shape)
    else:
        vecs = []
        for i in tqdm(range(0, len(chunks), 32)):
            batch = []
            for c in chunks[i:i + 32]:
                batch.append(c["text"])
            vecs += embed(batch)
        emb = np.array(vecs, dtype

In [5]:
import inspect

print("===== SCENARIO 검색 함수 =====")
print(inspect.getsource(sce_rag.search))

print("\n===== 초기화 관련 변수 =====")

for name in [
    "CARDS_URL",
    "EMBED_MODEL",
    "_model",
    "_collection",
    "model",
    "collection"
]:
    if hasattr(sce_rag, name):
        value = getattr(sce_rag, name)

        if name in ("_model", "_collection", "model", "collection"):
            print(f"{name}: {'초기화 전' if value is None else type(value).__name__}")
        else:
            print(f"{name}: {value}")
    else:
        print(f"{name}: 정의되지 않음")

===== SCENARIO 검색 함수 =====
def search(question, state, k=3, use_filter=True):
    # Chroma에서 유사도 순으로 전부 가져온 뒤, 지금 볼 수 있는 카드만 남기고 앞에서 k개를 돌려준다
    q_vec = _model.encode("query: " + question, normalize_embeddings=True)
    res = _collection.query(query_embeddings=[q_vec.tolist()], n_results=len(chunks))

    hits = []
    for cid, dist in zip(res["ids"][0], res["distances"][0]):
        if use_filter and not can_search(card_by_id[cid], state):
            continue
        hits.append((cid, round(1 - dist, 3)))      # (카드 id, 유사도)
    return hits[:k]


===== 초기화 관련 변수 =====
CARDS_URL: https://raw.githubusercontent.com/TRPG-masterbot/data/main/Kim/TheDrownedBell/game/data/cards.json
EMBED_MODEL: intfloat/multilingual-e5-small
_model: 초기화 전
_collection: 초기화 전
model: 정의되지 않음
collection: 정의되지 않음


In [6]:
sce_rag.load()

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

... (5 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

sce_rag 준비 완료: 카드 166장 중 검색 대상 112장


In [7]:
# 테스트 게임 상태
game_state = {
    "scene": 1,
    "cls": "fighter",
    "flags": set(),
    "visited": {"village_square"}
}

# 사용자 입력
query = "세라에게 마을에서 무슨 일이 있었는지 묻는다"

# 시나리오 검색
hits = sce_rag.search(
    question=query,
    state=game_state,
    k=3,
    use_filter=True
)

print("===== 검색 결과 =====")

for card_id, score in hits:
    print("카드 ID:", card_id)
    print("유사도:", score)
    print("내용:", sce_rag.card_text(card_id))
    print("-" * 50)

===== 검색 결과 =====
카드 ID: scn-g-rumors
유사도: 0.863
내용: 마을 사람들의 소문. 대장간 노인은 "종소리가 날마다 가까워진다"고 하고, 여관 주인은 밤에는 우물 곁을 지나지 말라고 이른다. 실종자 가족을 만나려면 세라를 찾아가라는 말이 어디서든 나온다. 소문의 절반은 과장이지만, 종소리가 우물 쪽에서만 난다는 말은 모두 같다.
--------------------------------------------------
카드 ID: scn-q1-npc-sera
유사도: 0.863
내용: 세라 — 실종자 토브의 아내이자 리나의 어머니. 30대 후반, 우물가에서 매일 밤을 새운다. 일행에게 매달리지 않고 낮은 목소리로 부탁한다. "찾을 수 없으면 그렇다고만 말해 줘요." 딸이 밤마다 들고 다니던 인형을 건네며 "이걸 보면 그 애가 알아볼 거예요"라고 한다. 남편과 딸이 사라진 날 모두 우물 쪽에서 종소리가 유난히 컸다고 기억한다.
--------------------------------------------------
카드 ID: scn-s1-desc
유사도: 0.847
내용: 에코 타운. 우물 속 소리가 메아리처럼 올라오던 마을이라 그렇게 불렸다. 채석장이 말라붙은 뒤로 스무 가구만 남은 국경 마을. 마을 한가운데 돌 뚜껑을 덮은 마른 우물이 있고, 그 주위만 풀이 자라지 않는다. 반년 전부터 우물물이 끊겼고, 밤이면 땅 밑에서 종소리가 희미하게 울린다. 종소리는 불규칙하고, 물이 출렁이는 듯 둔탁하다.
--------------------------------------------------


In [8]:
print("===== RULE RAG 설정 =====")

settings = [
    "OLLAMA_URL",
    "EMBED_MODEL",
    "GEN_MODEL",
    "RULES_URL",
    "OUT_PATH",
    "EMB_PATH",
    "COLLECTION",
    "TOP_K",
    "POOL_K"
]

for name in settings:

    value = getattr(rule_rag, name, "정의되지 않음")

    print(f"{name}: {value}")

===== RULE RAG 설정 =====
OLLAMA_URL: http://localhost:11434
EMBED_MODEL: bge-m3
GEN_MODEL: qwen2.5:14b
RULES_URL: 정의되지 않음
OUT_PATH: rules_new.jsonl
EMB_PATH: rules_new_emb.npy
COLLECTION: rule_rag
TOP_K: 5
POOL_K: 10


In [11]:
import requests
from pathlib import Path

print("===== RULE RAG 실행 환경 확인 =====")

# 1. Ollama 서버 연결
try:
    response = requests.get(
        f"{rule_rag.OLLAMA_URL}/api/tags",
        timeout=5
    )
    response.raise_for_status()

    print("Ollama 서버: 연결 성공")

    models = [
        item["name"]
        for item in response.json().get("models", [])
    ]

    print("설치된 모델:", models)

except requests.RequestException as e:
    print("Ollama 서버: 연결 실패")
    print("원인:", e)


# 2. 규칙 데이터 주소 확인
try:
    response = requests.get(
        rule_rag.RULES_URL,
        timeout=30,
        stream=True
    )

    print("\n규칙 데이터 HTTP 상태:", response.status_code)
    response.raise_for_status()

    print("규칙 데이터 주소: 정상")

    response.close()

except requests.RequestException as e:
    print("규칙 데이터 주소: 확인 실패")
    print("원인:", e)


# 3. 기존 임베딩 파일 확인
emb_path = Path(rule_rag.EMB_PATH)

print("\n임베딩 캐시 존재:", emb_path.exists())

if emb_path.exists():
    print("캐시 크기:", emb_path.stat().st_size, "bytes")

===== RULE RAG 실행 환경 확인 =====
Ollama 서버: 연결 실패
원인: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/tags (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7d081d7d8b90>: Failed to establish a new connection: [Errno 111] Connection refused'))

규칙 데이터 HTTP 상태: 200
규칙 데이터 주소: 정상

임베딩 캐시 존재: False


In [12]:
import requests

NEW_RULES_URL = (
    "https://raw.githubusercontent.com/"
    "TRPG-masterbot/data/main/Min/rules_new.jsonl"
)

response = requests.get(
    NEW_RULES_URL,
    timeout=30
)

print("HTTP 상태:", response.status_code)

if response.status_code == 200:

    rule_rag.RULES_URL = NEW_RULES_URL

    print("RULE RAG 데이터 주소 수정 완료")

else:

    print("해당 주소에서도 규칙 파일을 찾지 못했습니다.")

HTTP 상태: 200
RULE RAG 데이터 주소 수정 완료


In [13]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 93 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 2s (427 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [14]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [56]:
import subprocess
import time
import requests

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(5)

response = requests.get(
    "http://localhost:11434/api/tags",
    timeout=10
)

print("Ollama 서버 상태:", response.status_code)

Ollama 서버 상태: 200


In [57]:
!ollama pull bge-m3
!ollama pull qwen2.5:14b

In [58]:
import requests

OLLAMA_URL = "http://localhost:11434"

try:
    response = requests.get(
        f"{OLLAMA_URL}/api/tags",
        timeout=10
    )

    response.raise_for_status()

    print("Ollama 서버 연결 성공")
    print("설치된 모델:", [
        model["name"]
        for model in response.json().get("models", [])
    ])

except requests.RequestException as e:
    print("Ollama 서버 연결 실패:", e)

Ollama 서버 연결 성공
설치된 모델: ['qwen2.5:14b', 'bge-m3:latest']


In [23]:
import subprocess

result = subprocess.run(
    ["ollama", "pull", "bge-m3"],
    check=True
)

print("bge-m3 모델 설치 완료")

bge-m3 모델 설치 완료


In [24]:
import requests

response = requests.post(
    f"{OLLAMA_URL}/api/embed",
    json={
        "model": "bge-m3",
        "input": "플레이어가 적을 공격할 때 명중 판정을 한다."
    },
    timeout=120
)

response.raise_for_status()

data = response.json()

embedding = data["embeddings"][0]

print("임베딩 생성 성공")
print("벡터 차원:", len(embedding))
print("벡터 일부:", embedding[:5])

임베딩 생성 성공
벡터 차원: 1024
벡터 일부: [-0.041559953, 0.021928428, -0.058214534, 0.021115154, -0.006144573]


In [31]:
import requests

RULES_URL = (
    "https://raw.githubusercontent.com/"
    "TRPG-masterbot/data/main/Min/rules_new.jsonl"
)

# 1. 규칙 데이터 확인
response = requests.get(
    RULES_URL,
    timeout=30,
    stream=True
)

print("규칙 데이터 상태:", response.status_code)
response.raise_for_status()
response.close()

# 2. Ollama 연결 확인
ollama_response = requests.get(
    f"{rule_rag.OLLAMA_URL}/api/tags",
    timeout=10
)

ollama_response.raise_for_status()

installed_models = [
    model["name"]
    for model in ollama_response.json().get("models", [])
]

print("Ollama 연결 성공")
print("설치된 모델:", installed_models)

# 3. 임베딩 모델 설치 여부 확인
if not any(
    name.split(":")[0] == "bge-m3"
    for name in installed_models
):
    raise RuntimeError(
        "bge-m3 모델이 설치되지 않았습니다. "
        "먼저 ollama pull bge-m3를 실행하세요."
    )

# 4. RULE RAG 초기화
rule_rag.load()

print("RULE RAG 초기화 완료")

규칙 데이터 상태: 200
Ollama 연결 성공
설치된 모델: ['bge-m3:latest', 'qwen2.5:14b']
받음: rules_new.jsonl 1903588 bytes
검색 대상 청크: 2180


  0%|          | 0/69 [00:00<?, ?it/s]

새로 계산: (2180, 1024)
Chroma 벡터 수: 2180
준비 완료
RULE RAG 초기화 완료


In [32]:
import inspect

print("===== RULE 다운로드 함수 =====")
print(inspect.getsource(rule_rag.download))

print("\n===== RULE 설정 =====")

for name in [
    "RULES_URL",
    "BASE_URL",
    "OUT_PATH",
    "EMB_PATH",
    "EMBED_MODEL"
]:
    print(
        name,
        ":",
        getattr(rule_rag, name, "정의되지 않음")
    )

===== RULE 다운로드 함수 =====
def download(name):                 # 깃허브에서 파일 하나 받아 현재 폴더에 저장
    res = requests.get(f"{REPO}/{name}", timeout=60)
    res.raise_for_status()                         # 파일 이름이나 주소가 틀리면 여기서 멈춤
    with open(name, "wb") as f:
        f.write(res.content)
    print("받음:", name, len(res.content), "bytes")


===== RULE 설정 =====
RULES_URL : https://raw.githubusercontent.com/TRPG-masterbot/data/main/Min/rules_new.jsonl
BASE_URL : 정의되지 않음
OUT_PATH : rules_new.jsonl
EMB_PATH : rules_new_emb.npy
EMBED_MODEL : bge-m3


In [34]:
print("===== RULE RAG 초기화 검증 =====")

# 1. 규칙 청크 확인
print("규칙 청크 수:", len(rule_rag.chunks))

# 2. ChromaDB 벡터 개수 확인
print("ChromaDB 벡터 수:", rule_rag.collection.count())

# 3. BM25 초기화 확인
print("BM25 초기화:", rule_rag.bm25 is not None)

# 4. 규칙 조각 정보 확인
print("Parts 초기화:", rule_rag.parts is not None)

# 5. 전체 검증
assert len(rule_rag.chunks) > 0
assert rule_rag.collection.count() == len(rule_rag.chunks)
assert rule_rag.bm25 is not None
assert rule_rag.parts is not None

print("\nRULE RAG 초기화 검증 완료")

===== RULE RAG 초기화 검증 =====
규칙 청크 수: 2180
ChromaDB 벡터 수: 2180
BM25 초기화: True
Parts 초기화: True

RULE RAG 초기화 검증 완료


In [36]:
import requests

response = requests.get(
    f"{rule_rag.OLLAMA_URL}/api/tags",
    timeout=10
)

response.raise_for_status()

installed_models = [
    model["name"]
    for model in response.json()["models"]
]

print("===== Ollama 설치 모델 =====")

for name in installed_models:
    print(name)

print("\nRULE 임베딩 모델:", rule_rag.EMBED_MODEL)
print("RULE 번역 모델:", rule_rag.GEN_MODEL)

===== Ollama 설치 모델 =====
bge-m3:latest
qwen2.5:14b

RULE 임베딩 모델: bge-m3
RULE 번역 모델: qwen2.5:14b


In [51]:
import inspect
import rule_rag

print("===== RULE RAG retrieve() =====")
print("함수 매개변수:", inspect.signature(rule_rag.retrieve))
print("\n함수 코드:")
print(inspect.getsource(rule_rag.retrieve))

===== RULE RAG retrieve() =====
함수 매개변수: (question, k=5)

함수 코드:
def retrieve(question, k=TOP_K):                        # 최종 검색
    job1 = pool.submit(translate1, question)            # 번역1
    job2 = pool.submit(translate2, question)            # 번역2
    job0 = pool.submit(semantic, [question], POOL_K)    # 임베딩 : 원문 질문

    # 끝날 때까지 기다렸다가 결과 받기
    en1 = job1.result()
    en2 = job2.result()
    first = job0.result()

    pair = semantic2([en1, en2], POOL_K)             # 임베딩: 번역1, 2를 한 번에

    lists = [
        first,                                       # 임베딩: 원문 질문
        pair[0],                                     # 임베딩: 번역1
        pair[1],                                     # 임베딩: 번역2
        keyword(en1, POOL_K),                        # 키워드: 번역1
        keyword(en2, POOL_K),                        # 키워드: 번역2
    ]
    return merge(lists, SEARCH_WEIGHTS, k), [question, en1, en2]    # (청크 5개, 검색에 쓴 글 3개)



In [59]:
# RULE RAG 초기화 여부 확인
assert rule_rag.chunks is not None, "RULE RAG가 초기화되지 않았습니다."
assert rule_rag.collection is not None, "ChromaDB가 초기화되지 않았습니다."
assert rule_rag.bm25 is not None, "BM25가 초기화되지 않았습니다."

# 플레이어 질문
query = "근접 공격을 할 때 명중 판정은 어떻게 하나요?"

# RULE RAG 검색
rule_hits, translated_queries = rule_rag.retrieve(
    query,
    3
)

print("===== 검색어 =====")

for i, text in enumerate(translated_queries, start=1):
    print(f"{i}. {text}")


print("\n===== RULE RAG 검색 결과 =====")

for i, hit in enumerate(rule_hits, start=1):

    print(f"\n[{i}] 검색 결과")

    print("제목:", hit["meta"].get("title"))
    print("유형:", hit["meta"].get("type"))
    print("내용:", hit["text"][:500])
    print("-" * 60)

===== 검색어 =====
1. 근접 공격을 할 때 명중 판정은 어떻게 하나요?
2. Melee Attack Hit Determination
3. melee attack hit determination dnd rules

===== RULE RAG 검색 결과 =====

[1] 검색 결과
제목: Attack Notation
유형: rule
내용: Monsters > Parts of a Stat Block > Actions > Attack Notation
The entry for a monster’s attack identifies whether the attack is a melee or a ranged attack and then provides the attack roll’s bonus, its reach or range, and what happens on a hit. An attack is against one target unless its entry says otherwise. For details on different kinds of attacks, see “Playing the Game” and “Spells.” Hit. Any damage dealt or other effects that occur as a result of an attack hitting a target are described after
------------------------------------------------------------

[2] 검색 결과
제목: Melee Attacks
유형: rule
내용: Playing the Game > Combat > Melee Attacks
A melee attack allows you to attack a target within your reach. A melee attack typically uses a handheld weapon or an Unarmed Strike. Many monsters make melee

In [60]:
print("===== RULE RAG 검증 =====")

print("검색 결과 개수:", len(rule_hits))
print("검색어 개수:", len(translated_queries))

assert len(rule_hits) > 0, "검색 결과가 없습니다."

for hit in rule_hits:
    assert "text" in hit
    assert "meta" in hit

print("\nRULE RAG 검색 기능 정상 작동")

===== RULE RAG 검증 =====
검색 결과 개수: 3
검색어 개수: 3

RULE RAG 검색 기능 정상 작동


In [61]:
from pathlib import Path

Path("/content/trpg_ai").mkdir(parents=True, exist_ok=True)

In [62]:
%%writefile /content/trpg_ai/query_router.py

RULE_KEYWORDS = [
    "규칙", "판정", "주사위", "명중",
    "공격", "피해", "방어", "내성",
    "은신", "기술", "주문", "치명타",
    "민첩", "근력", "지혜", "굴림",
    "스킬", "전투", "기습"
]

SCENARIO_KEYWORDS = [
    "대화", "말한다", "묻는다", "거절",
    "받지 않", "수락", "보상", "돈",
    "마을", "광장", "우물", "던전",
    "고블린", "몬스터", "주민",
    "NPC", "상인", "세라", "할빅",
    "리나", "토브", "조사", "탐색",
    "단서", "소문", "방문"
]

def route_query(query):
    if not isinstance(query, str):
        raise TypeError("사용자 입력은 문자열이어야 합니다.")

    query = query.strip()

    if not query:
        raise ValueError("사용자 입력이 비어 있습니다.")

    normalized = query.lower()

    rule_matches = [
        word for word in RULE_KEYWORDS
        if word.lower() in normalized
    ]

    scenario_matches = [
        word for word in SCENARIO_KEYWORDS
        if word.lower() in normalized
    ]

    if rule_matches and scenario_matches:
        intent = "both"
    elif rule_matches:
        intent = "rule"
    elif scenario_matches:
        intent = "scenario"
    else:
        intent = "both"

    return {
        "player_input": query,
        "intent": intent,
        "use_rule": intent in ("rule", "both"),
        "use_scenario": intent in ("scenario", "both"),
        "matched_rule_keywords": rule_matches,
        "matched_scenario_keywords": scenario_matches
    }

Writing /content/trpg_ai/query_router.py


In [67]:
import sys
import importlib

PROJECT_DIR = "/content/trpg_ai"

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import query_router
importlib.reload(query_router)

result = query_router.route_query(
    "세라에게 실종자에 대해 묻는다"
)

print(result)

{'player_input': '세라에게 실종자에 대해 묻는다', 'intent': 'scenario', 'use_rule': False, 'use_scenario': True, 'matched_rule_keywords': [], 'matched_scenario_keywords': ['묻는다', '세라']}


In [70]:
import sys

PROJECT_DIR = str(Path("/content/trpg_ai"))

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import query_router
import rule_rag
import sce_rag

def search_rag(query, state, k=3):

    # 1. 입력 분류
    route = query_router.route_query(query)

    intent = route["intent"]

    # 2. 결과 저장 구조
    result = {
        "player_input": query,
        "intent": intent,
        "rule_results": [],
        "scenario_results": [],
        "translated_queries": []
    }

    # 3. RULE RAG 검색
    if route["use_rule"]:

        rule_hits, translated = rule_rag.retrieve(
            query,
            3
        )

        result["translated_queries"] = translated

        for hit in rule_hits:

            result["rule_results"].append({
                "source": "rule",
                "text": hit["text"],
                "score": None,
                "metadata": hit.get("meta", {})
            })

    # 4. SCENARIO RAG 검색
    if route["use_scenario"]:

        scenario_hits = sce_rag.search(
            question=query,
            state=state,
            k=k,
            use_filter=True
        )

        for card_id, score in scenario_hits:

            result["scenario_results"].append({
                "source": "scenario",
                "text": sce_rag.card_text(card_id),
                "score": score,
                "metadata": {
                    "card_id": card_id
                }
            })

    return result

In [75]:
%%writefile /content/trpg_ai/rag_bridge.py

import query_router
import rule_rag
import sce_rag


def search_rag(query, state, k=3):

    # 1. 입력 분류
    route = query_router.route_query(query)

    intent = route["intent"]

    # 2. 결과 구조
    result = {
        "player_input": route["player_input"],
        "intent": intent,
        "rule_results": [],
        "scenario_results": [],
        "translated_queries": []
    }

    # 3. RULE RAG 검색
    if route["use_rule"]:

        rule_hits, translated = rule_rag.retrieve(
            question = route["player_input"],
            k = k
        )

        result["translated_queries"] = translated

        for hit in rule_hits:

            result["rule_results"].append({
                "source": "rule",
                "text": hit["text"],
                "score": None,
                "metadata": hit.get("meta", {})
            })

    # 4. SCENARIO RAG 검색
    if route["use_scenario"]:

        scenario_hits = sce_rag.search(
            question=route["player_input"],
            state=state,
            k=k,
            use_filter=True
        )

        for card_id, score in scenario_hits:

            result["scenario_results"].append({
                "source": "scenario",
                "text": sce_rag.card_text(card_id),
                "score": score,
                "metadata": {
                    "card_id": card_id
                }
            })

    return result

Overwriting /content/trpg_ai/rag_bridge.py


In [76]:
import sys
import importlib

# 프로젝트 폴더
PROJECT_DIR = "/content/trpg_ai"

# 기존 RAG 모듈 폴더
RAG_DIR = "/content/rag"

# Python 검색 경로 등록
for path in [PROJECT_DIR, RAG_DIR]:
    if path not in sys.path:
        sys.path.insert(0, path)

# 통합 검색 모듈 불러오기
import rag_bridge

# 파일 수정 사항 반영
importlib.reload(rag_bridge)

print("rag_bridge.py 연결 성공")
print("통합 검색 함수:", rag_bridge.search_rag)

rag_bridge.py 연결 성공
통합 검색 함수: <function search_rag at 0x7d0513d18360>


In [77]:
import json

game_state = {
    "scene": 1,
    "cls": "fighter",
    "flags": set(),
    "visited": {"village_square"}
}

query = "세라에게 실종자에 대해 묻는다"

result = rag_bridge.search_rag(
    query=query,
    state=game_state,
    k=3
)

print(json.dumps(
    result,
    ensure_ascii=False,
    indent=2
))

{
  "player_input": "세라에게 실종자에 대해 묻는다",
  "intent": "scenario",
  "rule_results": [],
  "scenario_results": [
    {
      "source": "scenario",
      "text": "세라 — 실종자 토브의 아내이자 리나의 어머니. 30대 후반, 우물가에서 매일 밤을 새운다. 일행에게 매달리지 않고 낮은 목소리로 부탁한다. \"찾을 수 없으면 그렇다고만 말해 줘요.\" 딸이 밤마다 들고 다니던 인형을 건네며 \"이걸 보면 그 애가 알아볼 거예요\"라고 한다. 남편과 딸이 사라진 날 모두 우물 쪽에서 종소리가 유난히 컸다고 기억한다.",
      "score": 0.886,
      "metadata": {
        "card_id": "scn-q1-npc-sera"
      }
    },
    {
      "source": "scenario",
      "text": "마을 사람들의 소문. 대장간 노인은 \"종소리가 날마다 가까워진다\"고 하고, 여관 주인은 밤에는 우물 곁을 지나지 말라고 이른다. 실종자 가족을 만나려면 세라를 찾아가라는 말이 어디서든 나온다. 소문의 절반은 과장이지만, 종소리가 우물 쪽에서만 난다는 말은 모두 같다.",
      "score": 0.868,
      "metadata": {
        "card_id": "scn-g-rumors"
      }
    },
    {
      "source": "scenario",
      "text": "실종자 셋. 모두 지난 두 달 사이, 모두 밤에, 모두 우물 근처에서 마지막으로 목격됐다. 토브(채석공, 34세), 리나(토브의 딸, 11세), 가렌(떠돌이 땜장이, 나이 미상). 시신은 발견되지 않았다. 마을 사람들은 늑대나 도적 탓으로 돌리고 싶어 한다.",
      "score": 0.851,
      "metadata": {
        "card_i

In [78]:
test_queries = [
    "공격 판정은 어떻게 해?",
    "세라에게 실종자에 대해 묻는다",
    "마을 광장에서 은신 판정을 한다"
]

for query in test_queries:

    result = rag_bridge.search_rag(
        query=query,
        state=game_state,
        k=3
    )

    print("=" * 50)
    print("입력:", query)
    print("분류:", result["intent"])
    print("RULE 결과:", len(result["rule_results"]))
    print("SCENARIO 결과:", len(result["scenario_results"]))

입력: 공격 판정은 어떻게 해?
분류: rule
RULE 결과: 3
SCENARIO 결과: 0
입력: 세라에게 실종자에 대해 묻는다
분류: scenario
RULE 결과: 0
SCENARIO 결과: 3
입력: 마을 광장에서 은신 판정을 한다
분류: both
RULE 결과: 3
SCENARIO 결과: 3


In [79]:
import json

game_state = {
    "scene": 1,
    "cls": "fighter",
    "flags": set(),
    "visited": {"village_square"}
}

query = "세라에게 실종자에 대해 묻는다"

result = search_rag(
    query=query,
    state=game_state,
    k=3
)

print(json.dumps(
    result,
    ensure_ascii=False,
    indent=2
))

{
  "player_input": "세라에게 실종자에 대해 묻는다",
  "intent": "scenario",
  "rule_results": [],
  "scenario_results": [
    {
      "source": "scenario",
      "text": "세라 — 실종자 토브의 아내이자 리나의 어머니. 30대 후반, 우물가에서 매일 밤을 새운다. 일행에게 매달리지 않고 낮은 목소리로 부탁한다. \"찾을 수 없으면 그렇다고만 말해 줘요.\" 딸이 밤마다 들고 다니던 인형을 건네며 \"이걸 보면 그 애가 알아볼 거예요\"라고 한다. 남편과 딸이 사라진 날 모두 우물 쪽에서 종소리가 유난히 컸다고 기억한다.",
      "score": 0.886,
      "metadata": {
        "card_id": "scn-q1-npc-sera"
      }
    },
    {
      "source": "scenario",
      "text": "마을 사람들의 소문. 대장간 노인은 \"종소리가 날마다 가까워진다\"고 하고, 여관 주인은 밤에는 우물 곁을 지나지 말라고 이른다. 실종자 가족을 만나려면 세라를 찾아가라는 말이 어디서든 나온다. 소문의 절반은 과장이지만, 종소리가 우물 쪽에서만 난다는 말은 모두 같다.",
      "score": 0.868,
      "metadata": {
        "card_id": "scn-g-rumors"
      }
    },
    {
      "source": "scenario",
      "text": "실종자 셋. 모두 지난 두 달 사이, 모두 밤에, 모두 우물 근처에서 마지막으로 목격됐다. 토브(채석공, 34세), 리나(토브의 딸, 11세), 가렌(떠돌이 땜장이, 나이 미상). 시신은 발견되지 않았다. 마을 사람들은 늑대나 도적 탓으로 돌리고 싶어 한다.",
      "score": 0.851,
      "metadata": {
        "card_i

In [85]:
%%writefile /content/trpg_ai/main.py

import sys
import json
from pathlib import Path

# 1. 프로젝트 경로 설정
PROJECT_DIR = Path("/content/trpg_ai")
RAG_DIR = Path("/content/rag")

for path in [PROJECT_DIR, RAG_DIR]:
    if not path.exists():
        raise FileNotFoundError(f"폴더가 없습니다: {path}")

    if str(path) not in sys.path:
        sys.path.insert(0, str(path))


# 2. 기존 모듈 불러오기
import rule_rag
import sce_rag
from rag_bridge import search_rag
from handoff import build_llm_handoff


# 3. RULE 데이터 주소
RULES_URL = (
    "https://raw.githubusercontent.com/"
    "TRPG-masterbot/data/main/Min/rules_new.jsonl"
)


# 4. RAG 검색 환경 초기화
def initialize_rags():

    print("RULE RAG 초기화 중...")
    rule_rag.load()

    print("SCENARIO RAG 초기화 중...")
    sce_rag.load()

    print("두 RAG 검색 준비 완료")


# 5. 게임 상태
def create_game_state():

    return {
        "scene": 1,
        "cls": "fighter",
        "flags": set(),
        "visited": {"village_square"}
    }


# 6. 검색 실행
def run_game():

    initialize_rags()

    game_state = create_game_state()

    print("\n===== TRPG Dual RAG 테스트 =====")
    print("종료하려면 exit를 입력하세요.")

    while True:

        query = input("\n플레이어 행동: ").strip()

        if query.lower() == "exit":
            print("테스트 종료")
            break

        if not query:
            print("행동을 입력해주세요.")
            continue

        try:
            result = search_rag(
                query=query,
                state=game_state,
                k=3
            )
            payload = build_llm_handoff(
            search_result=result,
            game_state=game_state
            )

            print("\n===== LLM 전달 데이터 =====")

            print(json.dumps(
                payload,
                ensure_ascii=False,
                indent=2
            ))

        except Exception as e:
            print("검색 중 오류:", e)


if __name__ == "__main__":
    run_game()

Overwriting /content/trpg_ai/main.py


In [86]:
%%writefile /content/trpg_ai/handoff.py

def build_llm_handoff(search_result, game_state):

    # 1. 게임 상태를 JSON에 저장 가능한 형태로 변환
    clean_state = {
        "scene": game_state["scene"],
        "cls": game_state["cls"],
        "flags": sorted(game_state["flags"]),
        "visited": sorted(game_state["visited"])
    }

    # 2. RULE 검색 결과 정리
    rule_context = []

    for hit in search_result["rule_results"]:

        meta = hit.get("metadata", {})

        rule_context.append({
            "source": "rule",
            "title": meta.get("title"),
            "page": meta.get("page"),
            "text": hit["text"]
        })

    # 3. SCENARIO 검색 결과 정리
    scenario_context = []

    for hit in search_result["scenario_results"]:

        scenario_context.append({
            "source": "scenario",
            "card_id": hit["metadata"]["card_id"],
            "score": hit["score"],
            "text": hit["text"]
        })

    # 4. LLM 전달 데이터 생성
    payload = {
        "player_input": search_result["player_input"],
        "intent": search_result["intent"],
        "game_state": clean_state,
        "retrieved_context": {
            "rules": rule_context,
            "scenarios": scenario_context
        },
        "instructions": [
            "플레이어의 입력과 현재 게임 상태를 고려한다.",
            "규칙과 관련된 설명은 RULE 검색 결과를 근거로 한다.",
            "시나리오의 사건과 인물 정보는 SCENARIO 검색 결과를 근거로 한다.",
            "검색 결과에 없는 사실을 확정적으로 만들어내지 않는다.",
            "검색 결과가 부족하면 임의로 규칙이나 사건을 확정하지 않는다.",
            "플레이어에게 공개 가능한 정보만 응답에 사용한다."
        ]
    }

    return payload

Overwriting /content/trpg_ai/handoff.py


In [90]:
import sys
import json
import importlib

PROJECT_DIR = "/content/trpg_ai"

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import handoff
importlib.reload(handoff)

# 이전 단계에서 만든 게임 상태
game_state = {
    "scene": 1,
    "cls": "fighter",
    "flags": set(),
    "visited": {"village_square"}
}

# 기존 통합 검색 함수 호출
search_result = rag_bridge.search_rag(
    query="세라에게 실종자에 대해 묻는다",
    state=game_state,
    k=3
)

# LLM 전달용 데이터 생성
payload = handoff.build_llm_handoff(
    search_result=search_result,
    game_state=game_state
)

print(json.dumps(
    payload,
    ensure_ascii=False,
    indent=2
))

{
  "player_input": "세라에게 실종자에 대해 묻는다",
  "intent": "scenario",
  "game_state": {
    "scene": 1,
    "cls": "fighter",
    "flags": [],
    "visited": [
      "village_square"
    ]
  },
  "retrieved_context": {
    "rules": [],
    "scenarios": [
      {
        "source": "scenario",
        "card_id": "scn-q1-npc-sera",
        "score": 0.886,
        "text": "세라 — 실종자 토브의 아내이자 리나의 어머니. 30대 후반, 우물가에서 매일 밤을 새운다. 일행에게 매달리지 않고 낮은 목소리로 부탁한다. \"찾을 수 없으면 그렇다고만 말해 줘요.\" 딸이 밤마다 들고 다니던 인형을 건네며 \"이걸 보면 그 애가 알아볼 거예요\"라고 한다. 남편과 딸이 사라진 날 모두 우물 쪽에서 종소리가 유난히 컸다고 기억한다."
      },
      {
        "source": "scenario",
        "card_id": "scn-g-rumors",
        "score": 0.868,
        "text": "마을 사람들의 소문. 대장간 노인은 \"종소리가 날마다 가까워진다\"고 하고, 여관 주인은 밤에는 우물 곁을 지나지 말라고 이른다. 실종자 가족을 만나려면 세라를 찾아가라는 말이 어디서든 나온다. 소문의 절반은 과장이지만, 종소리가 우물 쪽에서만 난다는 말은 모두 같다."
      },
      {
        "source": "scenario",
        "card_id": "scn-s1-clue-missing",
        "score": 0.851,
        "text": "실종자 셋. 모두 지난 두 달 사이, 모두 밤에, 모

In [97]:
import rag_bridge
import handoff

# 테스트할 게임 상태
game_state = {
    "scene": 1,
    "cls": "fighter",
    "flags": set(),
    "visited": {"village_square"}
}

# 테스트 입력과 확인할 핵심 대상
test_cases = [
    {
        "query": "세라에게 실종자에 대해 묻는다",
        "target": "세라"
    },
    {
        "query": "할빅에게 돈을 받지 않는다",
        "target": "할빅"
    },
    {
        "query": "마을 우물에 대해 조사한다",
        "target": "우물"
    },
    {
        "query": "리나에 대해 물어본다",
        "target": "리나"
    }
]


def test_scenario_relevance(test_cases, state, k=5):

    for case in test_cases:

        query = case["query"]
        target = case["target"]

        print("\n" + "=" * 65)
        print("플레이어 입력:", query)
        print("확인할 대상:", target)

        # 기존 통합 검색 실행
        result = rag_bridge.search_rag(
            query=query,
            state=state,
            k=k
        )

        print("Query Router 분류:", result["intent"])

        scenario_hits = result["scenario_results"]

        if not scenario_hits:
            print("SCENARIO 검색 결과 없음")
            continue

        # 검색 결과 순서대로 검사
        for rank, hit in enumerate(scenario_hits, start=1):

            card_id = hit["metadata"]["card_id"]
            score = hit["score"]
            text = hit["text"]

            # 지정한 대상이 검색된 텍스트에 있는지 확인
            target_found = target.lower() in text.lower()

            print(f"\n[{rank}] 카드 ID: {card_id}")
            print("유사도:", score)
            print("핵심 대상 포함:", target_found)
            print("검색 내용:", text[:350])

        print("\n검색 카드 수:", len(scenario_hits))


test_scenario_relevance(
    test_cases=test_cases,
    state=game_state,
    k=5
)


플레이어 입력: 세라에게 실종자에 대해 묻는다
확인할 대상: 세라
Query Router 분류: scenario

[1] 카드 ID: scn-q1-npc-sera
유사도: 0.886
핵심 대상 포함: True
검색 내용: 세라 — 실종자 토브의 아내이자 리나의 어머니. 30대 후반, 우물가에서 매일 밤을 새운다. 일행에게 매달리지 않고 낮은 목소리로 부탁한다. "찾을 수 없으면 그렇다고만 말해 줘요." 딸이 밤마다 들고 다니던 인형을 건네며 "이걸 보면 그 애가 알아볼 거예요"라고 한다. 남편과 딸이 사라진 날 모두 우물 쪽에서 종소리가 유난히 컸다고 기억한다.

[2] 카드 ID: scn-g-rumors
유사도: 0.868
핵심 대상 포함: True
검색 내용: 마을 사람들의 소문. 대장간 노인은 "종소리가 날마다 가까워진다"고 하고, 여관 주인은 밤에는 우물 곁을 지나지 말라고 이른다. 실종자 가족을 만나려면 세라를 찾아가라는 말이 어디서든 나온다. 소문의 절반은 과장이지만, 종소리가 우물 쪽에서만 난다는 말은 모두 같다.

[3] 카드 ID: scn-s1-clue-missing
유사도: 0.851
핵심 대상 포함: False
검색 내용: 실종자 셋. 모두 지난 두 달 사이, 모두 밤에, 모두 우물 근처에서 마지막으로 목격됐다. 토브(채석공, 34세), 리나(토브의 딸, 11세), 가렌(떠돌이 땜장이, 나이 미상). 시신은 발견되지 않았다. 마을 사람들은 늑대나 도적 탓으로 돌리고 싶어 한다.

[4] 카드 ID: scn-v8-rumor-bell
유사도: 0.832
핵심 대상 포함: False
검색 내용: 소문 — 종소리. 마을 사람들은 지하의 종이 사람을 불러 간다고 수군거린다. 실종자 셋이 밤마다 우물 쪽으로 걸어간 것이 그 증거라고들 한다. 종소리가 들리는 밤에만 사람이 사라진다는 말도 있다.

[5] 카드 ID: scn-s1-desc
유사도: 0.802
핵심 대상 포함: False
검색 내용: 에코 타운. 우물 속 소리가 메아리처럼 올라오던 마을이

In [ ]:
%run /content/trpg_ai/main.py

RULE RAG 초기화 중...
받음: rules_new.jsonl 1903588 bytes
검색 대상 청크: 2180
저장된 임베딩 사용: (2180, 1024)
Chroma 벡터 수: 2180
준비 완료
SCENARIO RAG 초기화 중...
sce_rag 준비 완료: 카드 166장 중 검색 대상 112장
두 RAG 검색 준비 완료

===== TRPG Dual RAG 테스트 =====
종료하려면 exit를 입력하세요.
